# 🏦 SmartCredit — Intelligent Loan Default & Risk Scoring System
### End-to-End FinTech Machine Learning System (Lectures 01 – 28 Integration)

**Author:** Shivam Keshari  
**Domain:** Banking Analytics & FinTech Credit Risk  
**Core Pipeline:** Data Extraction $\to$ Preprocessing $\to$ Feature Engineering $\to$ Model Training (Logistic, Decision Tree with CCP, Random Forest) $\to$ Evaluation $\to$ Interactive Web App

---

## 1. Environment Setup & Library Imports

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay
)

plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries successfully imported!')

## 2. Data Ingestion & Hygiene Introspection
We load the official historical banking dataset containing 1,000 credit applications across 20 demographic and financial attributes.

In [ ]:
data_path = os.path.join('..', 'data', 'raw', 'loan_data.csv')
df = pd.read_csv(data_path)
print(f'Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns')
df.head()

In [ ]:
# Missing value inspection
missing = df.isnull().sum()
print('Columns with missing values:')
print(missing[missing > 0])

## 3. Data Cleaning & Preprocessing Pipeline
- Drop rows with unlabelled target `Loan_Approved`
- Impute numeric attributes with column median (outlier-robust)
- Impute categorical attributes with column mode
- Encode binary target (Yes $\to$ 1, No $\to$ 0)

In [ ]:
# Step 1: Drop rows where target is missing
df_clean = df.dropna(subset=['Loan_Approved']).copy()

# Step 2: Impute features
num_cols = df_clean.select_dtypes(include='number').columns
cat_cols = df_clean.select_dtypes(exclude='number').columns

for c in num_cols:
    if df_clean[c].isnull().any():
        df_clean[c] = df_clean[c].fillna(df_clean[c].median())

for c in cat_cols:
    if df_clean[c].isnull().any():
        df_clean[c] = df_clean[c].fillna(df_clean[c].mode()[0])

# Step 3: Encode target
df_clean['Loan_Approved'] = df_clean['Loan_Approved'].map({'Yes': 1, 'No': 0}).astype(int)

print(f'Cleaned dataset shape: {df_clean.shape}')
print(f'Remaining NaNs: {df_clean.isnull().sum().sum()}')
print('Target class distribution:\n', df_clean['Loan_Approved'].value_counts())

## 4. Exploratory Data Analysis (EDA)
Statistical visualizations uncovering the key behavioral indicators separating approved borrowers from high-risk defaulters.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Income distribution by approval
sns.kdeplot(data=df_clean, x='Applicant_Income', hue='Loan_Approved', common_norm=False, fill=True, ax=axes[0, 0], palette=['#e74c3c', '#2ecc71'])
axes[0, 0].set_title('Applicant Income Distribution by Approval', fontweight='bold')

# 2. Credit Score vs Approval status
sns.boxplot(data=df_clean, x='Loan_Approved', y='Credit_Score', ax=axes[0, 1], palette=['#e74c3c', '#2ecc71'])
axes[0, 1].set_xticklabels(['Rejected / Default', 'Approved'])
axes[0, 1].set_title('Credit Score Distribution by Loan Status', fontweight='bold')

# 3. DTI Ratio vs Loan Amount scatter
sns.scatterplot(data=df_clean, x='DTI_Ratio', y='Loan_Amount', hue='Loan_Approved', alpha=0.7, ax=axes[1, 0], palette=['#e74c3c', '#2ecc71'])
axes[1, 0].set_title('DTI Ratio vs Loan Amount', fontweight='bold')

# 4. Approval Rate by Employment Status
approval_by_emp = df_clean.groupby('Employment_Status')['Loan_Approved'].mean().reset_index()
sns.barplot(data=approval_by_emp, x='Employment_Status', y='Loan_Approved', ax=axes[1, 1], palette='Blues_d')
axes[1, 1].set_title('Approval Rate by Employment Status', fontweight='bold')
axes[1, 1].set_ylabel('Approval Rate')

plt.tight_layout()
plt.show()

## 5. Domain Feature Engineering
1. **Total Income:** Combined household earning capacity: $I_{\text{total}} = I_{\text{applicant}} + I_{\text{coapplicant}}$
2. **Income-to-Loan Ratio:** Financial safety margin: $R_{\text{income}} = \frac{I_{\text{total}}}{L}$
3. **Credit Tiers:** Non-linear categorization into Poor (<580), Fair (580-669), Good (670-739), and Excellent (740+)
4. **Financial Stress Flags:** High DTI ($>0.5$), Low Savings ($<5000$), High Loan-to-Income ($>5$)
5. **One-Hot Encoding:** Categorical attributes encoded with `drop_first=True` to prevent multicollinearity

In [ ]:
df_eng = df_clean.copy()

# 1. Income derivations
df_eng['total_income'] = df_eng['Applicant_Income'] + df_eng['Coapplicant_Income']
df_eng['income_loan_ratio'] = (df_eng['total_income'] / df_eng['Loan_Amount']).replace([np.inf, -np.inf], np.nan).fillna(0)

# 2. Credit tiers
bins = [-np.inf, 580, 670, 740, np.inf]
labels = ['Poor', 'Fair', 'Good', 'Excellent']
df_eng['credit_tier'] = pd.cut(df_eng['Credit_Score'], bins=bins, labels=labels, right=False)

# 3. Risk flags
df_eng['high_dti'] = (df_eng['DTI_Ratio'] > 0.5).astype(int)
df_eng['low_savings'] = (df_eng['Savings'] < 5000).astype(int)
loan_to_inc = (df_eng['Loan_Amount'] / df_eng['total_income']).replace([np.inf, -np.inf], np.nan).fillna(0)
df_eng['high_loan_ratio'] = (loan_to_inc > 5).astype(int)

# 4. Drop non-informative ID
if 'Applicant_ID' in df_eng.columns:
    df_eng.drop(columns=['Applicant_ID'], inplace=True)

# 5. One-hot encoding
cat_cols_to_encode = ['Employment_Status', 'Marital_Status', 'Loan_Purpose', 'Property_Area', 'Education_Level', 'Gender', 'Employer_Category', 'credit_tier']
df_model = pd.get_dummies(df_eng, columns=cat_cols_to_encode, drop_first=True)

bool_cols = df_model.select_dtypes(include='bool').columns
df_model[bool_cols] = df_model[bool_cols].astype(int)

print(f'Final engineered feature matrix: {df_model.shape[0]} rows × {df_model.shape[1]} features')

## 6. Supervised Machine Learning Pipeline
We compare three classical architectures:
1. **Logistic Regression** (Linear baseline with Sigmoid link function $\sigma(z) = \frac{1}{1 + e^{-z}}$)
2. **Decision Tree Classifier with Cost-Complexity Pruning** (Leo Breiman's minimal cost-complexity criterion $R_\alpha(T) = R(T) + \alpha |T|$)
3. **Random Forest Classifier** (Ensemble bagging with variance reduction across 200 de-correlated trees)

In [ ]:
X = df_model.drop(columns=['Loan_Approved'])
y = df_model['Loan_Approved']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train set: {X_train.shape[0]} samples')
print(f'Test set:  {X_test.shape[0]} samples')

In [ ]:
# 1. Logistic Regression
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

# 2. Decision Tree with Cost-Complexity Pruning
full_tree = DecisionTreeClassifier(class_weight='balanced', random_state=42)
full_tree.fit(X_train, y_train)
path = full_tree.cost_complexity_pruning_path(X_train, y_train)
alphas = [a for a in path.ccp_alphas if a >= 0]
if len(alphas) > 20:
    alphas = alphas[::len(alphas)//20]

best_alpha = 0.0
best_score = 0.0
for a in alphas:
    score = cross_val_score(DecisionTreeClassifier(ccp_alpha=a, class_weight='balanced', random_state=42), X_train, y_train, cv=5, scoring='accuracy').mean()
    if score > best_score:
        best_score = score
        best_alpha = a

dt = DecisionTreeClassifier(ccp_alpha=best_alpha, class_weight='balanced', random_state=42)
dt.fit(X_train, y_train)
print(f'Optimal Tree CCP Alpha: {best_alpha:.6f} (CV Accuracy: {best_score:.4f})')

# 3. Random Forest Classifier
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)
print('All 3 models successfully fitted!')

## 7. Model Evaluation & Benchmark Comparison

In [ ]:
models = {'Logistic Regression': lr, 'Decision Tree (Pruned)': dt, 'Random Forest': rf}
metrics = []

for name, m in models.items():
    y_pred = m.predict(X_test)
    y_prob = m.predict_proba(X_test)[:, 1]
    metrics.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1_Score': f1_score(y_test, y_pred),
        'ROC_AUC': roc_auc_score(y_test, y_prob)
    })

metrics_df = pd.DataFrame(metrics).set_index('Model')
metrics_df

In [ ]:
# ROC Curves Comparison
plt.figure(figsize=(9, 6))
for name, m in models.items():
    y_prob = m.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Guess')
plt.xlabel('False Positive Rate (FPR)', fontsize=12)
plt.ylabel('True Positive Rate (Recall)', fontsize=12)
plt.title('Receiver Operating Characteristic (ROC) Benchmark', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.show()

In [ ]:
# Top 10 Feature Importance from Random Forest
feat_imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 5))
sns.barplot(x=feat_imp.values, y=feat_imp.index, palette='viridis')
plt.title('Top 10 Driving Features for Credit Approval (Random Forest)', fontsize=14, fontweight='bold')
plt.xlabel('Gini Feature Importance', fontsize=12)
plt.show()

## 8. Real-Time Inference Demo (Simulating New Applicant)
We test our production pipeline on a hypothetical new credit applicant.

In [ ]:
sample_applicant = X_test.iloc[0:1]
pred = rf.predict(sample_applicant)[0]
proba = rf.predict_proba(sample_applicant)[0]

print('=== PREDICTION INFERENCE RESULT ===')
print(f'Decision: {"APPROVED ✅" if pred == 1 else "REJECTED / HIGH RISK ❌"}')
print(f'Approval Probability : {proba[1]*100:.2f}%')
print(f'Default Risk Factor  : {proba[0]*100:.2f}%')
if proba[0] < 0.3:
    print('Risk Tier: 🟢 LOW RISK (Eligible for Prime Interest Rates)')
elif proba[0] < 0.6:
    print('Risk Tier: 🟡 MEDIUM RISK (Requires Additional Collateral)')
else:
    print('Risk Tier: 🔴 HIGH RISK (Declined / Excessive Exposure)')